### Convert spreadsheet rows to Markdown documents

This cell creates one `.md` file for each row with nonblank text in both `Abstract` and `Claims`. Each field is labeled, for example, `**Abstract:** ...`, so the column name stays close to its text.

This approach assumes each row is a self-contained record. If rows only make sense together, create one Markdown file per worksheet instead, to keep those related rows in the same document. Note: if the ingestion chunker splits a long row, it may still separate a field label from some of its text; preserving labels in every chunk requires the chunker to keep or add the field label when splitting.

In [1]:
from pathlib import Path
import pandas as pd

# This must be the path inside the Coder workspace, not the C:\ path on your PC.
source = Path("/home/coder/advanced-RAG/advanced-RAG/my_corpus/Patent Dataset.XLSX")

if not source.is_file():
    raise FileNotFoundError(f"Excel file not found at: {source}")

# Read every worksheet in the workbook.
sheets = pd.read_excel(source, sheet_name=None, dtype=str)

# Confirm every worksheet has the columns needed for filtering.
required_columns = {"Abstract", "Claims"}
for sheet_name, df in sheets.items():
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"Worksheet {sheet_name!r} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

# Make one Markdown document for each eligible spreadsheet row.
corpus_folder = source.parent / "patent_text"
corpus_folder.mkdir(parents=True, exist_ok=True)

# Remove documents generated by this cell on an earlier run, so old rows
# cannot remain in the folder and accidentally be ingested.
for extension in ("txt", "md"):
    for old_file in corpus_folder.glob(f"sheet_*_row_*.{extension}"):
        old_file.unlink()

written = 0
total_rows = 0
skipped_rows = 0

for sheet_number, (sheet_name, df) in enumerate(sheets.items(), start=1):
    df = df.fillna("")
    total_rows += len(df)

    has_abstract = df["Abstract"].astype("string").str.strip().ne("")
    has_claims = df["Claims"].astype("string").str.strip().ne("")
    eligible_rows = df.loc[has_abstract & has_claims]
    skipped_rows += len(df) - len(eligible_rows)

    # Keep the original spreadsheet row number in the output filename.
    for row_index, row in eligible_rows.iterrows():
        spreadsheet_row = int(row_index) + 2  # Row 1 contains the column headers.

        fields = [
            f"**{column}:** {str(value).strip()}"
            for column, value in row.items()
            if str(value).strip()
        ]

        markdown = (
            f"# Patent record\n\n"
            f"- Worksheet: {sheet_name}\n"
            f"- Spreadsheet row: {spreadsheet_row}\n\n"
            + "\n\n".join(fields)
            + "\n"
        )

        filename = f"sheet_{sheet_number:02d}_row_{spreadsheet_row:06d}.md"
        (corpus_folder / filename).write_text(markdown, encoding="utf-8")
        written += 1

print(
    f"Created {written} Markdown documents; skipped {skipped_rows} of "
    f"{total_rows} rows missing Abstract or Claims."
)
print(f"Documents are in: {corpus_folder}")

# Show a preview so you can check the conversion.
markdown_files = sorted(corpus_folder.glob("*.md"))
if markdown_files:
    print(f"\nPreview of {markdown_files[0].name}:\n")
    print(markdown_files[0].read_text(encoding="utf-8")[:1000])

Created 1000 Markdown documents; skipped 0 of 1000 rows missing Abstract or Claims.
Documents are in: /home/coder/advanced-RAG/advanced-RAG/my_corpus/patent_text

Preview of sheet_01_row_000002.md:

# Patent record

- Worksheet: sheet1
- Spreadsheet row: 2

**Number:** 1

**Application Number:** US14/631607

**Publication Date:** 2015-10-20

**Application Date:** 2015-02-25

**Title:** Remote synchronization of pin-pad records with a central transactions database

**Abstract:** A method of remotely synchronizing pin-pad terminals involves a gateway receiving from an acquirer server an authorization code for a transaction authorized using one of a plurality of pin-pad terminals, and saving in a database particulars of the authorized transaction. Each terminal has a memory storing particulars of at least one authorized transaction authorized using the terminal, and a running total value of all transactions authorized using the pin-pad terminal. The gateway receives a request to close out

### Convert spreadsheet rows to documents

This cell reads every worksheet and creates one text document for each row that has nonblank text in both `Abstract` and `Claims`. Rows missing either value are skipped before the documents are used for ingestion.

```python
## Optional: If you want to create a text corpus instead of Markdown, uncomment the following code.
from pathlib import Path
import pandas as pd

# This must be the path inside the Coder workspace, not the C:\ path on your PC.
source = Path("/home/coder/advanced-RAG/advanced-RAG/my_corpus/Patent Dataset.XLSX")

if not source.is_file():
    raise FileNotFoundError(f"Excel file not found at: {source}")

# Read every worksheet in the workbook.
sheets = pd.read_excel(source, sheet_name=None, dtype=str)

# Confirm every worksheet has the columns needed for filtering.
required_columns = {"Abstract", "Claims"}
for sheet_name, df in sheets.items():
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"Worksheet {sheet_name!r} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

# Make one readable text document for each eligible spreadsheet row.
corpus_folder = source.parent / "patent_text"
corpus_folder.mkdir(parents=True, exist_ok=True)

# Remove documents generated by this cell on an earlier run, so rows excluded
# now cannot remain in the folder and accidentally be ingested.
for old_file in corpus_folder.glob("sheet_*_row_*.txt"):
    old_file.unlink()

written = 0
total_rows = 0
skipped_rows = 0

for sheet_number, (sheet_name, df) in enumerate(sheets.items(), start=1):
    df = df.fillna("")
    total_rows += len(df)

    has_abstract = df["Abstract"].astype("string").str.strip().ne("")
    has_claims = df["Claims"].astype("string").str.strip().ne("")
    eligible_rows = df.loc[has_abstract & has_claims]
    skipped_rows += len(df) - len(eligible_rows)

    for row_number, (_, row) in enumerate(eligible_rows.iterrows(), start=1):
        fields = [
            f"{column}: {str(value).strip()}"
            for column, value in row.items()
            if str(value).strip()
        ]

        text = f"Worksheet: {sheet_name}\n" + "\n".join(fields)
        filename = f"sheet_{sheet_number:02d}_row_{row_number:06d}.txt"
        (corpus_folder / filename).write_text(text, encoding="utf-8")
        written += 1

print(
    f"Created {written} text documents; skipped {skipped_rows} of "
    f"{total_rows} rows missing Abstract or Claims."
)
print(f"Documents are in: {corpus_folder}")

# Show a preview so you can check the conversion.
text_files = sorted(corpus_folder.glob("*.txt"))
if text_files:
    print(f"\nPreview of {text_files[0].name}:\n")
    print(text_files[0].read_text(encoding="utf-8")[:1000])
```

# RI #1: Bring your own corpus

This notebook builds a small evaluation set from a folder of your documents and ranks RI #1's
retrieval arms on it. Your corpus has no labeled questions, so the questions are generated: each
one is written from a statement quoted out of a document, and the quote's character offsets are
its gold evidence, which any chunker can be scored against.

Four code cells do the work: point the notebook at a folder, generate a set on your models,
score the arms, and report the spend. With `RUN_EXPENSIVE = False` and no folder named, the
notebook uses ten ContractNLI contracts as a stand-in and reads a reference set built on Vector's
on-prem models, with no model call. Name your folder and set the flag to `True` to build your
own.

The method and its limits are in
`../special_topics/synthetic_question_sets/01_build_and_trust_a_question_set.ipynb`. In short: on
ContractNLI, generated labels ranked these arms close to the human ranking (Kendall's tau 0.86 on
nDCG@10, the normalized discounted cumulative gain at rank 10), their absolute scores ran about 0.14 high, and they favored small chunks. The table
at the end gives an order of arms, and its scores are not comparable with scores on another set.

## 1. Point it at a folder

`FOLDER` is a directory of `.txt`, `.md`, or `.pdf` files, subfolders included. PDFs are
read through their text layer, so a scanned PDF needs optical character recognition (OCR) first, and a document whose answers
sit in tables reads better parsed with Docling, as RI #3 does. Web pages need converting to
`.md` or `.txt` first, and a bilingual PDF laid out in two columns reads as interleaved English and
French, so fetch the single-language HTML or text version where the publisher offers one.
`MAX_DOCUMENTS` takes the first files in sorted path order, so raise it to cover the whole folder. `CONFIG_FILE` is a copy of
`aieng-advanced-rag/aieng/advanced_rag/evaluation/synthetic/team-config.example.yaml` with your
domain, your users, and a few real questions as style examples. Without one, the notebook uses a
generic configuration at small sizes.

The generator is your generation model and the judge is your judging model (the shared
`gemini-3.8-flash` for paid families), as `choose_models` returns them. The checker comes from a
different family than the generator, so the round trip does not check a model against its own
habits: `gpt-6-luna` for a Google generator, `gemini-3.5-flash-lite` for any other paid generator, and a different on-prem Qwen model for an on-prem generator. To choose another,
pass `roles=Roles.from_models(MODELS, checker="<model_name>")` to `byoc.generate`.

```python
# This is an example only; Markdown cells don't execute Python.

from pathlib import Path
import pandas as pd
import byoc

from aieng.advanced_rag import spend
from aieng.advanced_rag.cache.paths import root as cache_root
from aieng.advanced_rag.evaluation.synthetic import Document
from aieng.advanced_rag.evaluation.synthetic.replay import write_beir
from aieng.advanced_rag.evaluation.synthetic.validation import chunks_for
from aieng.advanced_rag.models import choose_models

RUN_EXPENSIVE = True

source = Path(
    "/home/coder/advanced-RAG/advanced-RAG/my_corpus/Patent Dataset.XLSX"
)
CONFIG_FILE = Path(
    "/home/coder/advanced-RAG/advanced-RAG/my_corpus/patent_questions.yaml"
)

# Start small to test. Raise to 1000 to load up to 1000 eligible patents.
MAX_DOCUMENTS = 20

if not source.is_file():
    raise FileNotFoundError(f"Excel file not found at: {source}")

# openpyxl is the Excel-reading engine.
sheets = pd.read_excel(
    source,
    sheet_name=None,
    dtype=str,
    engine="openpyxl",
)

required_columns = {"Abstract", "Claims"}
documents = []
total_rows = 0
skipped_rows = 0

for sheet_name, df in sheets.items():
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"Worksheet {sheet_name!r} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    df = df.fillna("")
    total_rows += len(df)

    for row_index, row in df.iterrows():
        abstract = str(row["Abstract"]).strip()
        claims = str(row["Claims"]).strip()

        if not abstract or not claims:
            skipped_rows += 1
            continue

        excel_row = int(row_index) + 2  # Assumes row 1 contains column headers.
        application_number = str(row.get("Application Number", "")).strip()
        title = str(row.get("Title", "")).strip()

        # Keep every populated column as labeled text for the notebook's retriever.
        fields = [
            f"{column}: {str(value).strip()}"
            for column, value in row.items()
            if str(value).strip()
        ]

        text = f"Worksheet: {sheet_name}\n" + "\n\n".join(fields)
        doc_id = f"{sheet_name}-row-{excel_row}"

        documents.append(
            Document(
                doc_id=doc_id,
                title=(
                    f"{application_number} — {title}"
                    if application_number and title
                    else application_number or title or doc_id
                ),
                text=text,
            )
        )

        if len(documents) >= MAX_DOCUMENTS:
            break

    if len(documents) >= MAX_DOCUMENTS:
        break

if not documents:
    raise ValueError("No rows with nonblank Abstract and Claims were found.")

is_demo = False
MODELS = choose_models("openai")
config = byoc.config_for(CONFIG_FILE, is_demo)

print(
    f"Loaded {len(documents)} patent documents in memory; "
    f"skipped {skipped_rows} of {total_rows} rows missing Abstract or Claims."
)
print(f"First document: {documents[0].doc_id} — {documents[0].title}")
print(f"domain: {config.domain}; scope: {config.scope}")
```

In [2]:
import byoc
from aieng.advanced_rag import spend
from aieng.advanced_rag.cache.paths import root as cache_root
from aieng.advanced_rag.evaluation.synthetic.replay import write_beir
from aieng.advanced_rag.evaluation.synthetic.validation import chunks_for
from aieng.advanced_rag.models import choose_models


# Off reads the stand-in's reference set. On builds and scores on your models.
RUN_EXPENSIVE = True
# A folder of .txt, .md, or .pdf files. None uses ten ContractNLI contracts.
FOLDER = "/home/coder/advanced-RAG/advanced-RAG/my_corpus/patent_text"
# Your copy of team-config.example.yaml, or None.
CONFIG_FILE = "/home/coder/advanced-RAG/advanced-RAG/my_corpus/patent_questions.yaml"
# Read at most this many files, so a first run stays small.
MAX_DOCUMENTS = 20

MODELS = choose_models("anthropic")
documents, is_demo = byoc.read_corpus(FOLDER, limit=MAX_DOCUMENTS)
config = byoc.config_for(CONFIG_FILE, is_demo)
characters = sum(len(d.text) for d in documents)
print(f"{len(documents)} documents, {characters:,} characters{' (the ContractNLI stand-in)' if is_demo else ''}")
print(f"domain: {config.domain}; scope: {config.scope}; {config.passages_per_document} passages per document")

20 documents, 881,265 characters
domain: patent applications and inventions across technical fields; scope: corpus; 4 passages per document


## 2. Generate a small set

The build holds out a few documents for null questions, samples passages from the rest, asks the
generator for verbatim statements, and writes one question per statement. Each question then has
to pass a round trip (the checker answers from the quote alone and the judge compares that answer
with the reference), a uniqueness check against the nearest other passages, and deduplication.
Null questions are near-miss rewrites of accepted questions and questions written from the
held-out documents, kept only when the checker finds no answer in the top passages of the
indexed documents.

The label says where the set came from. With the flag off and the stand-in folder, it is the
reference set, followed by what building it on your models would cost. With the flag on and the
stand-in folder, the estimate prints first, and the build stops if it is over 10% of your key's
remaining budget, or over $1 when your key reports no budget and $300 is assumed. Pass
`confirm=True` to run it. Your own folder has no reference build to scale, so the label says
`no estimate` and the build starts at once. To cap a paid build, set `ADVANCED_RAG_SPEND_LIMIT_USD`
before you run the cell. Each model is held to 15 requests a minute: at the small sizes, a folder
of 48 regulations took 519 calls and 22 minutes.

In [3]:
result, generated = byoc.generate(documents, config, MODELS, live=RUN_EXPENSIVE, demo=is_demo)
summary = generated.summary()
funnel = generated.meta["funnel"]
print()
print(
    f"{summary['answerable']} answerable and {summary['null']} null questions; held out: {', '.join(generated.meta['documents']['held_out'])}"
)
print(f"{'kind':<16}{'stage':<18}{'entered':>8}{'passed':>8}")
for row in funnel["stages"]:
    print(f"{row['kind']:<16}{row['stage']:<18}{row['entered']:>8}{row['passed']:>8}")
print()
for q in generated.answerable[:3]:
    print(
        f"{q.question}\n  -> {q.reference}  [{q.evidence[0].doc_id}, characters {q.evidence[0].start} to {q.evidence[0].end}]"
    )

question-set running on claude-haiku-5-5: no estimate (no reference build to scale)


/home/coder/advanced-RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10481.94it/s]
19:46:36 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
19:46:37 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
Task was destroyed but it is pending!
task: <Task pending name='Task-493' coro=<LoggingWorker._worker_loop() done, defined at /home/coder/advanced-RAG/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:110> wait_for=<Future can

question-set your run: claude-haiku-5-5
             742 calls, 591,316 prompt + 59,672 completion tokens
             742 calls, 591,316 in + 59,672 out tokens on claude-haiku-5-5, gemini-3.5-flash-lite, gemini-3.8-flash, est. $0.23 over 3 models

142 answerable and 20 null questions; held out: sheet_01_row_000012.md, sheet_01_row_000018.md, sheet_01_row_000020.md
kind            stage              entered  passed
single_hop      quote_located          111     106
single_hop      question_written       106     102
single_hop      round_trip             102      86
single_hop      uniqueness              86      86
single_hop      deduplicated            86      80
near_miss_null  null_written            20      20
near_miss_null  null_check              20      10
held_out_null   quote_located           32      25
held_out_null   null_written            20      20
held_out_null   null_check              20      16
near_miss_null  deduplicated            10      10
held_out_null   dedu

## 3. Score RI #1's arms on the set

The eight arms are the ones the ContractNLI validation compared: BM25, dense
`bge-small-en-v1.5`, hybrid reciprocal rank fusion at dense weights 0.5 and 0.2, dense with a
MiniLM reranker, and a shuffled control at 1,200-character chunks, and BM25 and dense at
600-character chunks. They index the documents the set was built over, minus the held-out ones.
A set built with `scope: corpus` is searched across the whole folder, and `scope: document`
searches each question's own document, as RI #1 does for contracts.

Scoring is local, with no model call. With the flag off it reads cached scores for a set the cache
already holds. With it on it embeds the chunks on the CPU, which takes a few minutes for a small
folder. MRR is mean reciprocal rank. The last column is the top arm's nDCG@10 minus this arm's, with a
95% paired bootstrap interval over the questions. An interval that includes zero means this set
cannot tell the two arms apart.

The cell also writes the set and its 1,200-character chunking in the layout of Benchmarking Information Retrieval (BEIR), so tools that read
BEIR's query relevance judgments, or qrels (`pytrec_eval`, `ranx`, LlamaIndex's `EmbeddingQAFinetuneDataset`), can score your own
retrievers against it.

In [4]:
scores = byoc.score(result, generated, documents, live=RUN_EXPENSIVE)
mean = scores["mean"]
versus = scores["versus_best_ndcg@10"]
print(
    f"{scores['questions']} answerable questions over {scores['documents_indexed']} indexed documents{', corpus-wide' if scores['corpus_wide'] else ', each question in its own document'}"
)
print(f"{'rank':<6}{'arm':<24}{'nDCG@10':>9}{'recall@3':>10}{'MRR':>7}   nDCG@10 against {versus['best']}")
order = sorted(mean["ndcg@10"], key=lambda arm: -mean["ndcg@10"][arm])
for rank, arm in enumerate(order, 1):
    gap = versus["arms"].get(arm)
    interval = f"{gap['mean_difference']:+.3f} [{gap['low']:+.3f}, {gap['high']:+.3f}]" if gap else "top arm"
    print(
        f"{rank:<6}{arm:<24}{mean['ndcg@10'][arm]:>9.3f}{mean['recall@3'][arm]:>10.3f}{mean['mrr'][arm]:>7.3f}   {interval}"
    )

indexed = set(generated.meta["documents"]["indexed"])
beir_dir = cache_root() / "byoc-demo" / "beir" / result.digest[:12]
written = write_beir(
    generated.variant("plain"), chunks_for([d for d in documents if d.doc_id in indexed], 1200, 200), beir_dir
)
print(
    f"\nBEIR export: {written['queries']} queries, {written['corpus']} chunks, {written['qrels']} qrels rows, under the local cache"
)

arm scores: scoring the set 1c7fd3bf169e locally


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1938.84it/s]


80 answerable questions over 17 indexed documents, corpus-wide
rank  arm                       nDCG@10  recall@3    MRR   nDCG@10 against hybrid-rrf-0.5@1200
1     hybrid-rrf-0.5@1200         0.590     0.473  0.629   top arm
2     hybrid-rrf-0.2@1200         0.573     0.470  0.604   +0.016 [-0.020, +0.055]
3     dense+rerank@1200           0.546     0.440  0.562   +0.043 [-0.013, +0.096]
4     bm25@1200                   0.504     0.438  0.517   +0.086 [+0.023, +0.147]
5     dense@1200                  0.492     0.382  0.513   +0.098 [+0.043, +0.157]
6     bm25@600                    0.447     0.329  0.494   +0.142 [+0.078, +0.209]
7     dense@600                   0.389     0.271  0.471   +0.200 [+0.144, +0.256]
8     shuffled-control@1200       0.010     0.000  0.010   +0.579 [+0.498, +0.661]

BEIR export: 100 queries, 752 chunks, 199 qrels rows, under the local cache


On the stand-in contracts, the reranker arm comes first, and the two hybrids and both BM25 arms
sit within an interval of it that includes zero, so 27 questions cannot separate those five. Both
dense arms and the shuffled control are separated from it. On ContractNLI, generated labels also
put the reranker first and the control last, and gave 21 of the 22 arm pairs that human labels
separated the same sign.

Three cautions from the ContractNLI validation carry over to your corpus. Generated labels score
every arm higher than human questions would. They rate 600-character dense retrieval about level
with 1,200-character dense, where human labels put it 0.10 nDCG@10 lower. A set generated this
way says nothing about which answer model is better.

To grow the set, raise `MAX_DOCUMENTS` or `passages_per_document` in your configuration file.
The cost line in section 2 scales with both. Before you rely on the ranking, read 20 to 30
questions with someone who knows the documents and count the correct reference answers.

[`07_levers_and_answers_on_your_corpus.ipynb`](07_levers_and_answers_on_your_corpus.ipynb) picks up
this set: it adds SPLADE to the arms, compares each with BM25, and runs answering with abstention
on the answerable and null questions.

In [5]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
this run, proxy usage $0.00 (team key: includes teammates' calls in the same window)
key budget $300.00, spent $0.45, left $299.55 (100%)
